# <font color="#418FDE" size="6.5" uppercase>**Metadata Callbacks**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Enable and configure metadata routing for estimators and pipelines. 
- Route sample weights and groups through compatible workflows. 
- Explain callback registration and propagation concepts in supported estimators. 


## **1. Routing Setup**

### **1.1. Enable Routing**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_30/Lecture_A/image_01_01.jpg?v=1784039636" width="250">



>* Enable pipelines to recognize extra metadata
>* Route metadata only to compatible components

>* Pipelines coordinate metadata across individual steps
>* Routing improves clarity and reliable experimentation

>* Metadata routing defines clear component responsibilities
>* Careful setup improves transparency and reproducibility



In [ ]:
#@title Python Code - Enable Routing

# This example enables metadata routing in scikit-learn.
# Sample weights are routed through a pipeline.
# The output confirms weighted fitting behavior.

import numpy as np
import sklearn
from sklearn import set_config
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Metadata routing must be enabled before configuring requests.
set_config(enable_metadata_routing=True)

# A small dataset keeps the routing example easy to inspect.
features, target = make_classification(
    n_samples=120,
    n_features=4,
    n_informative=3,
    n_redundant=0,
    random_state=42,
)

# These weights make class one observations more influential.
sample_weight = np.where(target == 1, 5.0, 1.0)

# Validate that every row has exactly one metadata weight.
if features.shape[0] != sample_weight.shape[0]:
    raise ValueError("Each sample must have one weight.")

# The scaler ignores weights, while the classifier requests them.
scaler = StandardScaler().set_fit_request(sample_weight=False)
classifier = LogisticRegression(max_iter=300, random_state=42)
classifier = classifier.set_fit_request(sample_weight=True)

# The pipeline can now route sample_weight to compatible steps.
model = Pipeline(
    steps=[("scale", scaler), ("classify", classifier)]
)

# Fit once while passing metadata to the pipeline itself.
model.fit(features, target, sample_weight=sample_weight)

# Compare with an unweighted fit to show routing changed training.
unweighted_model = Pipeline(
    steps=[("scale", StandardScaler()), ("classify", LogisticRegression(max_iter=300, random_state=42))]
)

unweighted_model.fit(features, target)

weighted_intercept = model.named_steps["classify"].intercept_[0]
unweighted_intercept = unweighted_model.named_steps["classify"].intercept_[0]

print("scikit-learn version:", sklearn.__version__)
print("Metadata routing enabled:", sklearn.get_config()["enable_metadata_routing"])
print("Classifier requested sample_weight for fit:", True)
print("Weighted intercept:", round(weighted_intercept, 3))
print("Unweighted intercept:", round(unweighted_intercept, 3))



### **1.2. Fit Metadata Requests**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_30/Lecture_A/image_01_02.jpg?v=1784039634" width="250">



>* Declare training metadata each estimator accepts
>* Route, ignore, or reject metadata safely

>* Send metadata only to compatible steps
>* Clarify each component’s training role

>* Make fit expectations explicit and inspectable
>* Improve reproducibility, maintenance, and pipeline robustness



In [ ]:
#@title Python Code - Fit Metadata Requests

# This example configures fit metadata routing.
# Sample weights should reach only the classifier.
# Printed routing details confirm the request.

import sklearn
from sklearn import set_config
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Metadata routing must be enabled before requests are used.
set_config(enable_metadata_routing=True)

# A small dataset keeps the routing example easy to inspect.
features, target = make_classification(
    n_samples=80,
    n_features=4,
    random_state=42,
)

# Heavier weights make class one more influential during fitting.
sample_weights = 1 + target * 4

# The scaler ignores weights, while the classifier requests them.
scaler = StandardScaler().set_fit_request(sample_weight=False)
classifier = LogisticRegression(max_iter=200, random_state=42)
classifier = classifier.set_fit_request(sample_weight=True)

# The pipeline can now route fit metadata deliberately.
pipeline = Pipeline([
    ("scale", scaler),
    ("model", classifier),
])

# This single fit call supplies metadata to the routing system.
pipeline.fit(features, target, sample_weight=sample_weights)

# Inspecting requests shows which step receives sample weights.
scale_request = pipeline.named_steps["scale"].get_metadata_routing()
model_request = pipeline.named_steps["model"].get_metadata_routing()

print("scikit-learn version:", sklearn.__version__)
print("Scaler fit request:", scale_request.fit.requests["sample_weight"])
print("Classifier fit request:", model_request.fit.requests["sample_weight"])
print("Pipeline fitted with one shared sample_weight argument.")



### **1.3. Score Metadata Requests**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_30/Lecture_A/image_01_03.jpg?v=1784039638" width="250">



>* Scoring can use weights, groups, or metadata
>* Estimators declare scoring metadata handling choices

>* Pipelines route scoring metadata through relevant steps
>* Weights and groups support fairer evaluation

>* Make scoring metadata explicit and reliable
>* Align reported scores with evaluation goals



In [ ]:
#@title Python Code - Score Metadata Requests

# This example configures score metadata routing.
# Sample weights change the reported pipeline score.
# The output compares weighted and unweighted scoring.

import numpy as np
import sklearn
from sklearn import set_config
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Metadata routing must be enabled before requests are configured.
set_config(enable_metadata_routing=True)

# Create a small classification dataset for a fast demonstration.
features, target = make_classification(
    n_samples=120, n_features=4, n_informative=3, n_redundant=0, random_state=42
)

# Give the first twenty observations extra importance during scoring.
sample_weight = np.ones(target.shape[0])
sample_weight[:20] = 5.0

# Build one simple pipeline with preprocessing and classification.
model = LogisticRegression(max_iter=200, random_state=42)
model.set_score_request(sample_weight=True)

# The pipeline forwards score metadata to the final estimator.
pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("model", model),
])

# Fit uses ordinary data, while score receives requested metadata.
pipeline.fit(features, target)
unweighted_score = pipeline.score(features, target)
weighted_score = pipeline.score(features, target, sample_weight=sample_weight)

# Print concise results that show the routing effect.
print("scikit-learn version:", sklearn.__version__)
print("Score request for sample_weight: True")
print("Unweighted accuracy:", round(unweighted_score, 3))
print("Weighted accuracy:", round(weighted_score, 3))



## **2. Routing Metadata**

### **2.1. Sample Weight Routing**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_30/Lecture_A/image_02_01.jpg?v=1784039630" width="250">



>* Weights show each example’s importance
>* Routing sends weights to compatible workflow steps

>* Pass weights only to compatible workflow steps
>* Declare weight handling to prevent silent mistakes

>* Weights reflect fairness and operational priorities
>* Routing preserves meaning across compatible workflow steps



In [ ]:
#@title Python Code - Sample Weight Routing

# This example routes sample weights through a pipeline.
# Metadata routing sends weights only to compatible steps.
# Weighted training changes the fitted classifier result.

import numpy as np
import sklearn
from sklearn import set_config
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Enable scikit-learn metadata routing for this session.
set_config(enable_metadata_routing=True)

# Create a small imbalanced classification dataset.
features, labels = make_classification(
    n_samples=300,
    n_features=4,
    n_informative=3,
    n_redundant=0,
    weights=[0.85, 0.15],
    random_state=42,
)

# Give the rare class more influence during fitting.
sample_weights = np.where(labels == 1, 6.0, 1.0)

# Split features, labels, and weights together.
split_data = train_test_split(
    features,
    labels,
    sample_weights,
    test_size=0.30,
    stratify=labels,
    random_state=42,
)

X_train, X_test, y_train, y_test, w_train, w_test = split_data

# Validate that every training row has one weight.
if X_train.shape[0] != w_train.shape[0]:
    raise ValueError("Each training row needs exactly one sample weight.")

# Build a pipeline with one transformer and one classifier.
weighted_model = Pipeline(
    [
        ("scale", StandardScaler()),
        ("classify", LogisticRegression(max_iter=500, random_state=42)),
    ]
)

# The scaler ignores weights, while the classifier requests them.
weighted_model.named_steps["scale"].set_fit_request(sample_weight=False)
weighted_model.named_steps["classify"].set_fit_request(sample_weight=True)

# Fit once while routing sample weights through the pipeline.
weighted_model.fit(X_train, y_train, sample_weight=w_train)

# Fit a comparison model without sample weights.
plain_model = Pipeline(
    [
        ("scale", StandardScaler()),
        ("classify", LogisticRegression(max_iter=500, random_state=42)),
    ]
)

plain_model.fit(X_train, y_train)

# Compare ordinary and weighted evaluation results.
plain_predictions = plain_model.predict(X_test)
weighted_predictions = weighted_model.predict(X_test)

plain_accuracy = accuracy_score(y_test, plain_predictions)
weighted_accuracy = accuracy_score(y_test, weighted_predictions)
weighted_score = accuracy_score(y_test, weighted_predictions, sample_weight=w_test)

print("scikit-learn version:", sklearn.__version__)
print("Training rows:", X_train.shape[0])
print("Rare-class training weight:", float(w_train[y_train == 1][0]))
print("Plain test accuracy:", round(plain_accuracy, 3))
print("Routed weighted-model accuracy:", round(weighted_accuracy, 3))
print("Weighted evaluation accuracy:", round(weighted_score, 3))



### **2.2. Group Metadata Routing**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_30/Lecture_A/image_02_02.jpg?v=1784039628" width="250">



>* Track related records across workflow steps
>* Prevent leakage during validation and evaluation

>* Keep related records in the same fold
>* Send group labels only where requested

>* Workflow stages request only needed group context
>* Routing improves validation, model selection, and transparency



In [ ]:
#@title Python Code - Group Metadata Routing

# This example routes group labels through cross-validation.
# GroupKFold keeps related records in one fold.
# The output compares grouped and ordinary validation.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create small classification data with repeated group membership.
features, target = make_classification(
    n_samples=120, n_features=6, n_informative=4, random_state=42
)

# Each group represents related records from one source.
groups = np.repeat(np.arange(30), 4)

# Validate that every row has one group label.
if len(groups) != len(target):
    raise ValueError("Each sample must have exactly one group label.")

# Build a pipeline whose estimator ignores groups.
model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=500, random_state=42)
)

# GroupKFold requests groups when it creates validation folds.
group_splitter = GroupKFold(n_splits=5)
group_scores = cross_val_score(model, features, target, cv=group_splitter, groups=groups)

# Ordinary KFold does not know about group membership.
plain_splitter = KFold(n_splits=5, shuffle=True, random_state=42)
plain_scores = cross_val_score(model, features, target, cv=plain_splitter)

# Inspect one grouped split to confirm no group leakage.
train_rows, test_rows = next(group_splitter.split(features, target, groups))
shared_groups = set(groups[train_rows]).intersection(set(groups[test_rows]))

print("scikit-learn version:", sklearn.__version__)
print("Groups shared in first GroupKFold split:", len(shared_groups))
print("Mean accuracy with routed groups:", round(group_scores.mean(), 3))
print("Mean accuracy without groups:", round(plain_scores.mean(), 3))



### **2.3. Metadata Aliases**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_30/Lecture_A/image_02_03.jpg?v=1784039632" width="250">



>* Aliases translate metadata names across workflow components
>* They route weights and groups through pipelines

>* Aliases match metadata names across workflow components
>* They reduce reshaping and repeated parameter passing

>* Aliases direct metadata to the right components
>* Clear routing improves debugging and workflow maintenance



In [ ]:
#@title Python Code - Metadata Aliases

# This example demonstrates metadata aliases in routing.
# Aliases translate external names for estimator metadata.
# The output shows weighted fitting through an alias.

import sklearn
from sklearn import set_config
from sklearn.base import BaseEstimator
from sklearn.base import ClassifierMixin
from sklearn.pipeline import Pipeline

# Metadata routing must be enabled before requests are configured.
set_config(enable_metadata_routing=True)

class AliasDemoClassifier(BaseEstimator, ClassifierMixin):
    # This estimator records the metadata it receives during fitting.
    def fit(self, X, y, sample_weight=None):
        self.classes_ = sorted(set(y))
        self.received_weight_sum_ = sum(sample_weight)
        self.majority_class_ = self.classes_[0]
        return self

    # Prediction is simple because routing is the lesson here.
    def predict(self, X):
        return [self.majority_class_] * len(X)

# Small in-memory data keeps the routing behavior easy to inspect.
features = [[0.0], [1.0], [2.0], [3.0]]
target = [0, 0, 1, 1]
importance = [1.0, 1.0, 5.0, 5.0]

# The estimator asks for sample_weight under the external alias importance.
classifier = AliasDemoClassifier()
classifier = classifier.set_fit_request(sample_weight="importance")

# The pipeline forwards the alias to the estimator during fit.
pipeline = Pipeline([("model", classifier)])
pipeline.fit(features, target, importance=importance)

# The fitted estimator confirms that the alias delivered the weights.
fitted_model = pipeline.named_steps["model"]
print("scikit-learn version:", sklearn.__version__)
print("External metadata name: importance")
print("Estimator parameter name: sample_weight")
print("Sum received through alias:", fitted_model.received_weight_sum_)



## **3. Callback Routing**

### **3.1. Pipeline Callback Routing**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_30/Lecture_A/image_03_01.jpg?v=1784039640" width="250">



>* Callbacks go to relevant pipeline stages
>* Pipelines coordinate metadata for compatible steps

>* Route callbacks to relevant pipeline steps
>* Preserve meaning across nested components

>* Higher-level callbacks improve reusable workflow design
>* Estimator declarations guide deliberate callback propagation



In [ ]:
#@title Python Code - Pipeline Callback Routing

# This example shows pipeline callback routing.
# Metadata is forwarded only to compatible steps.
# The output confirms selective callback propagation.

import sklearn
from sklearn.base import BaseEstimator
from sklearn.base import ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris

# A tiny callback records which estimator received it.
class TrainingCallback:
    def __init__(self):
        self.events = []

    def record(self, step_name, message):
        self.events.append(step_name + ": " + message)

# This estimator accepts a callback during fitting.
class CallbackClassifier(BaseEstimator, ClassifierMixin):
    def fit(self, X, y, callback=None):
        self.classes_ = sorted(set(y))
        self.majority_class_ = max(self.classes_, key=list(y).count)
        if callback is not None:
            callback.record("model", "fit callback received")
        return self

    def predict(self, X):
        return [self.majority_class_] * len(X)

# Load a small packaged dataset for a quick demonstration.
iris = load_iris()
X = iris.data[:30]
y = iris.target[:30]

# Validate the simple shape assumption before fitting.
if len(X) != len(y):
    raise ValueError("Feature rows and labels must match.")

# The pipeline contains one transformer and one callback-aware model.
pipe = Pipeline(
    [("scale", StandardScaler()), ("model", CallbackClassifier())]
)

# The callback is routed only to the final model step.
callback = TrainingCallback()
pipe.fit(X, y, model__callback=callback)

# Print concise evidence of selective routing.
print("scikit-learn version:", sklearn.__version__)
print("Pipeline steps:", ", ".join(pipe.named_steps.keys()))
print("Callback events:", "; ".join(callback.events))
print("Scaler received callback: no")
print("Model received callback:", len(callback.events) == 1)



### **3.2. Registering Callbacks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_30/Lecture_A/image_03_02.jpg?v=1784039641" width="250">



>* Callbacks notify estimators during key workflow events
>* Routing controls what metadata callbacks receive

>* Define each callback’s metadata needs
>* Route only requested information to callbacks

>* Explicit registration clarifies callback scope
>* Metadata-aware rules improve debugging and reuse



### **3.3. Callback Propagation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_30/Lecture_A/image_03_03.jpg?v=1784039643" width="250">



>* Callbacks route through nested workflow components
>* Only compatible inner estimators receive them

>* Callbacks reach only relevant sub-estimators
>* Selective routing prevents misuse and confusion

>* Callbacks support monitoring, debugging, and reproducibility
>* Routing contracts prevent silent callback failures



# <font color="#418FDE" size="6.5" uppercase>**Metadata Callbacks**</font>


In this lecture, you learned to:
- Enable and configure metadata routing for estimators and pipelines. 
- Route sample weights and groups through compatible workflows. 
- Explain callback registration and propagation concepts in supported estimators. 

In the next Lecture (Lecture B), we will go over 'Inspection Displays'